# eph_06 — Behavioral comparison: RT encoding vs Sue model outputs

Compares per-unit RT encoding T-statistics with AIND behavioral model ("Sue") encodings:
outcome, Q-value (Qchosen), and baseline firing.

**Pipeline:**
1. Re-fit RT encoding, register in `PerUnitStatsRegistry`
2. Bulk-register all Sue T-columns via `register_sue`
3. Screen RT T-stats against all Sue columns (Spearman ρ of T-stat vectors)
4. `registry_compare_plot` for top-correlated Sue entries
5. T-scatter + polar histogram for key pairs (outcome, Q-chosen, baseline)

## 1. Setup

In [ ]:
%matplotlib inline
import contextlib, io
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from plotstyle import apply_style, PALETTE, style_ax, save_fig
apply_style()

In [ ]:
from pathlib import Path

if Path("/root/capsule").exists():
    ENV       = "codeocean"
    SCRATCH   = Path("/root/capsule/scratch")
    DATA_ROOT = Path("/root/capsule/data")
    FOR_LOCAL = SCRATCH / "for_local"
else:
    ENV       = "local"
    FOR_LOCAL = Path("/Users/mib/Documents/Code/kinematics_analysis/data/for_local")
    DATA_ROOT = FOR_LOCAL
    SCRATCH   = FOR_LOCAL.parent

FIG_DIR  = SCRATCH / "figures" / "eph_06_behavioral_comparison"
SAVE_FIG = False
print(f"ENV={ENV}  FOR_LOCAL={FOR_LOCAL}")

## 2. Data loading

In [ ]:
from data_loading import (
    load_session_quality_filter, filter_ephys_units, load_units_with_spike_times,
)
import pickle

if ENV == "codeocean":
    base_dirs = [SCRATCH / "session_analysis_mlk"]
    filtered_session_paths = load_session_quality_filter(base_dirs)
    with open(SCRATCH / "combined_unit_tbl.pkl", "rb") as f:
        combined_ephys_data = pickle.load(f)
    filtered_ephys = filter_ephys_units(combined_ephys_data, filtered_session_paths)
    ROOT_SCRATCH = str(DATA_ROOT / "LC-NE_scratch_data_1")
    units_with_spikes = load_units_with_spike_times(filtered_ephys, ROOT_SCRATCH)
else:
    filtered_ephys    = pd.read_pickle(FOR_LOCAL / "filtered_ephys.pkl")
    units_with_spikes = None
    base_dirs         = [FOR_LOCAL]
    print(f"Local dev: filtered_ephys {filtered_ephys.shape}")

In [ ]:
from ephys_utils import AnalysisConfig, build_all_counts_df

cfg = AnalysisConfig(
    align_key="goCue",
    count_window_s=(0.0, 0.2),
    baseline_window_s=(-1.0, 0.0),
    min_trials_per_group=20,
)
if ENV == "codeocean":
    all_counts_df = build_all_counts_df(units_with_spikes, cfg, base_dirs)
else:
    all_counts_df = pd.read_parquet(FOR_LOCAL / "all_counts_df.parquet")
print("all_counts_df:", all_counts_df.shape)

## 3. Imports

In [ ]:
from encoding_methods import AnalysisSpec, fit_encoding
from per_unit_stats_registry import PerUnitStatsRegistry
from encoding_plots import registry_compare_plot, tstat_hist
from aind_dynamic_foraging_behavior_video_analysis.ephys.tongue_ephys import get_session_prefix
import contextlib, io

## 4. Load Sue behavioral model table

In [ ]:
# Load Sue behavioral model encoding table
if ENV == "codeocean":
    sue_path = SCRATCH / "features_combined_beh_all.pkl"
else:
    sue_path = FOR_LOCAL / "features_combined_beh_all.pkl"

sue_features = pd.read_pickle(sue_path)
print("sue_features shape:", sue_features.shape)
print("T_ columns:", [c for c in sue_features.columns if str(c).startswith("T_")][:8], "...")

## 5. Fit RT encoding and register

In [ ]:
# Fit RT encoding (same spec as eph_01) to get per-unit T-stats
RT_SPEC = AnalysisSpec(
    name="ols_rt",
    predictor_col="reaction_time_firstmove",
    response_col="spike_count",
    method="ols",
    trial_query="reaction_time_firstmove > 0.05 and reaction_time_firstmove < 1.5",
    log_x=True, zscore_x=True,
    notes="ols: spike_count ~ log(RT)",
)
with contextlib.redirect_stdout(io.StringIO()):
    rt_result = fit_encoding(all_counts_df, RT_SPEC)
print(f"RT n_sig: {rt_result.n_sig()}")

In [ ]:
reg = PerUnitStatsRegistry(get_session_prefix=get_session_prefix, alpha=0.05)
reg.register(rt_result)

# Bulk-register all Sue T_ columns as "sue::{suffix}" entries
n_sue = reg.register_sue(
    sue_features,
    t_prefix="T_", p_prefix="p_", coef_prefix="coef_",
    registry_prefix="sue",
)
print(f"Registered {n_sue} Sue entries")
print(reg)

## 6. Screen: which Sue T-columns correlate most with RT encoding?

In [ ]:
# Screen RT T-stats against all Sue T_ columns via Spearman T-T correlation
screen = reg.screen(
    "ols_rt",
    source="sue",
    min_n=10,
    rank_by="abs_rho",
    top_n=20,
)
print("Top 20 Sue variables correlated with RT encoding T-stats:")
print(screen[["entry","n","rho","p","abs_rho","fisher_OR","fisher_p"]].to_string(index=False))

## 7. Registry compare plots for top Sue variables

In [ ]:
# Compare RT T-stats vs top Sue variables using registry_compare_plot.
# Take top 4 by |rho| with sufficient overlap.
top_entries = screen.query("n >= 20").head(4)["entry"].tolist()
print("Plotting comparisons for:", top_entries)

for sue_entry in top_entries:
    try:
        merged = reg.compare("ols_rt", sue_entry)
        fig = registry_compare_plot(
            merged, "ols_rt", sue_entry,
            title=f"RT encoding vs {sue_entry.replace('sue::','Sue: ')}",
        )
        save_fig(fig, f"compare_rt_vs_{sue_entry.replace('::', '_')}",
                 fig_dir=FIG_DIR, save=SAVE_FIG)
        plt.show()
    except Exception as e:
        print(f"  {sue_entry}: {e}")

## 8. Build merged table (sue_plus) for T-scatter

In [ ]:
# T-scatter: RT vs outcome encoding, using the merge directly.
# Builds sue_plus for cell-level scatter + polar histogram.
def _merge_rt_and_sue(rt_stats, sue_df, get_session_prefix_fn):
    """Inner join RT T-stats onto sue_df on (session_prefix, unit)."""
    def canon(x):
        try: return str(int(float(x)))
        except Exception: return str(x)

    rt = rt_stats[["session_prefix","unit","T","coef","sig_fdr"]].copy()
    rt["unit"] = rt["unit"].map(canon)
    rt = rt.rename(columns={"T":"T_rt", "coef":"coef_rt", "sig_fdr":"sig_rt"})

    sue = sue_df.copy()
    if "unit_id" in sue.columns and "unit" not in sue.columns:
        sue = sue.rename(columns={"unit_id":"unit"})
    sue["unit"] = sue["unit"].map(canon)
    if "session_prefix" not in sue.columns:
        sue["session_prefix"] = sue["session"].astype(str).map(get_session_prefix_fn)
    sue["session_prefix"] = sue["session_prefix"].astype(str)

    return sue.merge(rt, on=["session_prefix","unit"], how="inner")

sue_plus = _merge_rt_and_sue(rt_result.stats, sue_features, get_session_prefix)
print(f"sue_plus shape: {sue_plus.shape}")
print(f"T_rt non-null: {sue_plus['T_rt'].notna().sum()}")

## 9. T-scatter + polar histogram

Each scatter: x = RT T-stat, y = Sue T-stat for one behavioral variable.
Polar histogram: distribution of coefficient vector angles, showing whether units
co-modulated by RT and the behavioral variable tend toward a preferred direction.

In [ ]:
# T-scatter + polar histogram for key comparison pairs.
def plot_T_scatter_and_polar(df, t_x_col, t_y_col, coef_x_col, coef_y_col,
                              *, polar_bins=12, figsize=(11, 5), title=None):
    """Scatter of T-stats + polar histogram of coefficient vector angles."""
    from scipy.stats import spearmanr

    mask = (df[t_x_col].notna() & df[t_y_col].notna() &
            df[coef_x_col].notna() & df[coef_y_col].notna())
    d = df[mask].copy()
    if len(d) < 5:
        print(f"Skipping {title}: too few points ({len(d)})")
        return None

    rho, p = spearmanr(d[t_x_col], d[t_y_col])
    angles = np.arctan2(d[coef_y_col].to_numpy(dtype=float),
                        d[coef_x_col].to_numpy(dtype=float))

    fig, (ax_s, ax_p) = plt.subplots(1, 2, figsize=figsize)

    # scatter
    ax_s.scatter(d[t_x_col], d[t_y_col], s=8, alpha=0.4, color=PALETTE["neutral"])
    ax_s.axhline(0, color="black", lw=0.6, ls="--")
    ax_s.axvline(0, color="black", lw=0.6, ls="--")
    ax_s.set_xlabel(t_x_col)
    ax_s.set_ylabel(t_y_col)
    ax_s.set_title(f"T-scatter  rho={rho:.3f}  p={p:.2g}  n={len(d)}")
    style_ax(ax_s)

    # polar
    ax_p = plt.subplot(1, 2, 2, projection="polar")
    edges = np.linspace(-np.pi, np.pi, polar_bins + 1)
    counts, _ = np.histogram(angles, bins=edges)
    width = edges[1] - edges[0]
    ax_p.bar(edges[:-1] + width / 2, counts, width=width * 0.9,
             color=PALETTE["accent"], alpha=0.7)
    ax_p.set_title("Coef angle (rad)", va="bottom")

    fig.suptitle(title or f"{t_x_col} vs {t_y_col}", fontsize=11)
    plt.tight_layout()
    return fig

# Key pairs: RT vs outcome, RT vs Q-chosen
PAIRS = [
    ("T_rt", "T_outcome_com_ori", "coef_rt", "coef_outcome_com_ori"),
    ("T_rt", "T_Qchosen_com_ori", "coef_rt", "coef_Qchosen_com_ori"),
    ("T_rt", "T_baseline_hit_all","coef_rt", "coef_baseline_hit_all"),
]
for tx, ty, cx, cy in PAIRS:
    if tx in sue_plus.columns and ty in sue_plus.columns:
        fig = plot_T_scatter_and_polar(
            sue_plus, tx, ty, cx, cy,
            title=f"{tx} vs {ty}",
        )
        if fig:
            save_fig(fig, f"t_scatter_{tx}_vs_{ty}", fig_dir=FIG_DIR, save=SAVE_FIG)
            plt.show()
    else:
        print(f"Skipping {tx}/{ty}: columns not found")